In [1]:
import cobra
import pickle 
import numpy as np
from cobra import Reaction, Metabolite
from helper import flip_reverse_reactions

cobra_config = cobra.Configuration()
cobra_config.solver = "cplex"
cobra_config.tolerance = 1e-9

In [2]:
bm_rxns = {"Geobacter": "agg_GS13m", "Rhodoferax": "BIO_Rfer3"}
c_sources = ["EX_ac_e", "EX_mal-L_e", "EX_cit_e", "EX_fum_e"]


dimes = {"Rhodoferax": {
            "mEmC_1": {"uptakes": ["EX_fe2_e", "EX_fum_e"],
                        "secretion": ["EX_mal-L_e"]},
            "mEmC_3": {"uptakes": [ "EX_fe2_e", "EX_fum_e"],
                        "secretion": ["EX_mal-L_e"]},
            "mEmC_2": {"uptakes": [ "EX_fe2_e", "EX_cit_e"],
                      "secretion": ["EX_mal-L_e"]},
            "mEmC_4": {"uptakes": [ "EX_fe2_e", "EX_cit_e"],
                        "secretion": ["EX_mal-L_e"]},
            "mEmC_5": {"uptakes": [ "EX_fe2_e", "EX_mal-L_e"],
                        "secretion": []},
            "mEmC_6": {"uptakes": [ "EX_fe2_e", "EX_mal-L_e", "EX_co2_e"],
                      "secretion": []},
            "mE_1": {"uptakes": ["EX_ac_e", "EX_fe3_e"],
                    "secretion": []},
},
        "Geobacter": {
            "mEmC_1": {"uptakes": ["EX_nh4_e", "EX_mal-L_e"],
                      "secretion": ["EX_co2_e"]},
            "mEmC_2": {"uptakes": ["EX_nh4_e", "EX_mal-L_e"],
                      "secretion": ["EX_co2_e"]},
            "mEmC_3": {"uptakes": ["EX_n2_e", "EX_mal-L_e"],
                      "secretion": ["EX_co2_e"]},
            "mEmC_4": {"uptakes": ["EX_n2_e", "EX_mal-L_e"],
                      "secretion": ["EX_co2_e"]},
            "mEmC_5": {"uptakes": ["EX_n2_e", "EX_cit_e"],
                      "secretion": ["EX_mal-L_e", "EX_co2_e"]},
            "mEmC_6": {"uptakes": ["EX_n2_e", "EX_cit_e"],
                      "secretion": ["EX_mal-L_e", "EX_co2_e"]},
            "mE_1": {"uptakes": ["EX_n2_e", "EX_mal-L_e"],
                  "secretion": ["EX_ac_e", "EX_co2_e"]},
    }}

# from their github
def open_exchanges(model, organism):
    
    if organism == 'Geobacter':
        allowed = ['so4_e',
             'pi_e',
             'mg2_e',
             'k_e',
             'ca2_e',
             'fe3_e']
    elif organism == 'Rhodoferax':
        allowed = [
             'nh4_e',
             'pi_e',
             'so4_e',
             'o2_e'
            ]

    # we close demand reactions here, as the other functions do not care about the demands
    # close the demand reactions in Geobacter
    if organism == "Geobacter":
        to_remove = [rxn.id for rxn in model.reactions if rxn.id.startswith("DM_")]

        model.remove_reactions(to_remove)

    return allowed


def implement_dime(model, allowed, dime, c_sources, allow_secretions=True):
    # first, allow all exchanges as in the paper and turn off all secretions
    mu = model.slim_optimize() 
    print(f" * Growth rate before implementing DiME {mu:.2f}")

    to_remove = []
    for ex in model.exchanges:
        if ex.id in [ "EX_h2o_e",  "EX_h2_e",  "EX_h_e" ]:
            ex.bounds = (-1000, 1000)
            continue

        # dime should overwrite the allowed
        if ex.id in dime["secretion"]:
            ex.bounds = (0, 1000)
        elif (ex.id.replace("EX_", "") in allowed) or (ex.id in dime["uptakes"]):
            ex.bounds = (-1000, 0)
            if ex.id in c_sources:
                ex.bounds = (-10, 0)
        else:
            ex.bounds = (0, 1000)
            if not allow_secretions:
                to_remove.append(ex)

    model.remove_reactions(to_remove)
    cobra.manipulation.delete.prune_unused_metabolites(model)

    mu = model.slim_optimize() 
    print(f" * Growth rate after implementing DiME {mu:.2f}")

    return mu



In [3]:
def add_product_to_reaction(model, 
                            reaction_id, 
                            metabolite_id,
                            stoichiometry = 1.0,
                            name = ""):
    """
    Add a new product to an existing reaction in a COBRApy model.

    Parameters
    ----------
    model          : cobra.Model  – the model to modify
    reaction_id    : str          – ID of the reaction to update
    metabolite_id  : str          – ID of the metabolite to add as product
    stoichiometry  : float        – stoichiometric coefficient (positive = product)
    name           : str          – human-readable metabolite name (optional)
    """

    metabolite = Metabolite(
        id=metabolite_id,
        name=name,
        compartment="c"
    )
    reaction = model.reactions.get_by_id(reaction_id)
    reaction.add_metabolites({metabolite: stoichiometry})

In [4]:
reduced_models = {"Rhodoferax": {}, "Geobacter": {}}
dime_max_growth = {}
for modelname in reduced_models:
    print(f"\n***** {modelname} *****")
    reduced_models[modelname] = {}

    for dime_number, dime in dimes[modelname].items():
        print(f"Testing DiME {dime_number}")

        model = cobra.io.load_matlab_model(f"models/{modelname}.mat")
        allowed = open_exchanges(model, modelname)

        if modelname == "Geobacter":
            model.reactions.ATPM.bounds = (0.,1000.)

        to_remove = []
        for rxn in model.reactions:
            if abs(rxn.lower_bound) < 1e-15 and abs(rxn.upper_bound) < 1e-15:
                print(f"{rxn.id} removed\n")
                to_remove.append(rxn)
        model.remove_reactions(to_remove)
        cobra.manipulation.delete.prune_unused_metabolites(model)

        mu = implement_dime(model, allowed, dime, c_sources)      

        try:
            dime_max_growth[dime_number].append(mu)
        except KeyError:
            dime_max_growth[dime_number] = [mu]

        reduced_models[modelname][dime_number] = model


***** Rhodoferax *****
Testing DiME mEmC_1


No defined compartments in model model. Compartments will be deduced heuristically using regular expressions.
Using regular expression found the following compartments:c, e


 * Growth rate before implementing DiME 41.92
 * Growth rate after implementing DiME 0.67
Testing DiME mEmC_3


No defined compartments in model model. Compartments will be deduced heuristically using regular expressions.
Using regular expression found the following compartments:c, e


 * Growth rate before implementing DiME 41.92
 * Growth rate after implementing DiME 0.67
Testing DiME mEmC_2


...

FUMt4 removed

 * Growth rate before implementing DiME 35.65
 * Growth rate after implementing DiME 1.05
Testing DiME mEmC_6


No defined compartments in model model. Compartments will be deduced heuristically using regular expressions.
Using regular expression found the following compartments:c, e
/home/users/dszeliova/.conda/envs/comms2/lib/python3.11/site-packages/cobra/core/group.py:147: UserWarning: need to pass in a list
  warn("need to pass in a list")


FUMt4 removed

 * Growth rate before implementing DiME 35.65
 * Growth rate after implementing DiME 1.05
Testing DiME mE_1


No defined compartments in model model. Compartments will be deduced heuristically using regular expressions.
Using regular expression found the following compartments:c, e
/home/users/dszeliova/.conda/envs/comms2/lib/python3.11/site-packages/cobra/core/group.py:147: UserWarning: need to pass in a list
  warn("need to pass in a list")


FUMt4 removed

 * Growth rate before implementing DiME 35.65
 * Growth rate after implementing DiME 0.93


In [5]:
dime_max_growth

{'mEmC_1': [0.6698781291076985, 1.015500608059892],
 'mEmC_3': [0.6698781291076985, 0.9311962701878479],
 'mEmC_2': [1.0048171936615455, 1.015500608059892],
 'mEmC_4': [1.0048171936615455, 0.9311962701878479],
 'mEmC_5': [0.6698781291077002, 1.0514486861492156],
 'mEmC_6': [0.3253519821503404, 1.0514486861492156],
 'mE_1': [0.3694464604864135, 0.9311962701878479]}

In [6]:
reduced_models["Rhodoferax"]["mEmC_6"].summary()

Metabolite,Reaction,Flux,C-Number,C-Flux
fe2_e,EX_fe2_e,1.106E-05,0,0.00%
h_e,EX_h_e,28.3,0,0.00%
mal-L_e,EX_mal-L_e,10,4,100.00%
nh4_e,EX_nh4_e,14,0,0.00%
o2_e,EX_o2_e,2.364,0,0.00%
pi_e,EX_pi_e,0.303,0,0.00%
so4_e,EX_so4_e,0.08134,0,0.00%
Metabolite,Reaction,Flux,C-Number,C-Flux
h2o_e,EX_h2o_e,-30.1,0,0.00%
orot_e,EX_orot_e,-5.226,5,100.00%


In [7]:
new_names = []
simulated_mus = {}
for modelname in reduced_models:
    print(f"\n***** {modelname} *****")
    for dime_number, model in reduced_models[modelname].items():
        print(dime_number)
        simulated_mus[dime_number] = []

        dime = dimes[modelname][dime_number]
        min_mu = min(dime_max_growth[dime_number])  # lower growth rate is limiting

        for fraction in [0.99, 0.9, 0.8, 0.7, 0.6, 0.5, 0.4, 0.3]:
            model_modified = model.copy()
            set_mu = min_mu*fraction
            simulated_mus[dime_number].append(set_mu)
    
            # modify model such that constraints are part of matrix
            add_product_to_reaction(model_modified, 
                                    reaction_id=bm_rxns[modelname], 
                                    metabolite_id="fixed_growth",
                                    stoichiometry=1.0,
                                    name="fixed_growth")
    
            c_source = list(set(dime["uptakes"]).intersection(set(c_sources)))
            if len(c_source) > 1:
                print("More than one C source")
    
            model_modified.reactions.get_by_id(c_source[0]).lower_bound = -1000
    
            add_product_to_reaction(model_modified, 
                                    reaction_id=c_source[0], 
                                    metabolite_id="fixed_uptake",
                                    stoichiometry=-1.0,
                                    name="fixed_uptake")

            # reaction with values for the constraints
            # when flux 1 - constraints are active
            constraint_reaction =  Reaction("constraint_reaction")
            constraint_reaction.lower_bound = 1
            constraint_reaction.upper_bound = 1
            constraint_reaction.add_metabolites({
                model_modified.metabolites.fixed_growth: -set_mu,
                model_modified.metabolites.fixed_uptake: -10
            })

            # slack variable to allow inequality constraint for uptake
            slack_reaction = Reaction("slack_reaction")
            slack_reaction.lower_bound = 0
            slack_reaction.upper_bound = 1000
            slack_reaction.add_metabolites({
                model_modified.metabolites.fixed_uptake: 1
            })
            model_modified.add_reactions([constraint_reaction, slack_reaction])
    
            flip_reverse_reactions(model_modified)
    
            mu = model_modified.slim_optimize() 
            print(f" * Growth rate after preparation for ecmtool {mu:.2f}")
    
            new_name = f"{modelname}_{dime_number}_{round(set_mu, 2)}.xml"
            cobra.io.write_sbml_model(model_modified, f"models/{new_name}")
            new_names.append(new_name)


with open('tested_growth_rates.pkl', 'wb') as f:
    pickle.dump(simulated_mus, f)


***** Rhodoferax *****
mEmC_1
 * Growth rate after preparation for ecmtool 0.66
 * Growth rate after preparation for ecmtool 0.60
 * Growth rate after preparation for ecmtool 0.54
 * Growth rate after preparation for ecmtool 0.47
 * Growth rate after preparation for ecmtool 0.40
 * Growth rate after preparation for ecmtool 0.33
 * Growth rate after preparation for ecmtool 0.27
 * Growth rate after preparation for ecmtool 0.20
mEmC_3
 * Growth rate after preparation for ecmtool 0.66
 * Growth rate after preparation for ecmtool 0.60
 * Growth rate after preparation for ecmtool 0.54
 * Growth rate after preparation for ecmtool 0.47
 * Growth rate after preparation for ecmtool 0.40
 * Growth rate after preparation for ecmtool 0.33
 * Growth rate after preparation for ecmtool 0.27
 * Growth rate after preparation for ecmtool 0.20
mEmC_2
 * Growth rate after preparation for ecmtool 0.99
 * Growth rate after preparation for ecmtool 0.90
 * Growth rate after preparation for ecmtool 0.80
 * Gr

In [8]:
test = cobra.io.read_sbml_model("models/Geobacter_mEmC_5_0.66.xml")

In [9]:
test.reactions.agg_GS13m.lower_bound = 0.663179
test.objective = "EX_mal-L_e"
test.reactions.EX_fe2_e.bounds = (-2000, 2000)
test.reactions.EX_fe3_e.bounds = (-2000, 2000)
test.reactions.EX_h_e.bounds = (-2000, 2000)
test.reactions.FERCYT.bounds = (-2000, 2000)
fluxes = test.optimize()

test.summary()

Metabolite,Reaction,Flux,C-Number,C-Flux
ca2_e,EX_ca2_e,0.0006237,0,0.00%
fe3_e,EX_fe3_e,1132,0,0.00%
h2_e,EX_h2_e,591.6,0,0.00%
k_e,EX_k_e,0.1472,0,0.00%
mg2_e,EX_mg2_e,0.02121,0,0.00%
n2_e,EX_n2_e,2.251,0,0.00%
pi_e,EX_pi_e,1.179,0,0.00%
so4_e,EX_so4_e,0.1108,0,0.00%
Metabolite,Reaction,Flux,C-Number,C-Flux
fe2_e,EX_fe2_e,-1132,0,0.00%


In [10]:
for rxn, flux in fluxes.fluxes.items():
    if abs(flux) > 500:
        print(rxn, test.reactions.get_by_id(rxn).reaction, flux)

CYOR1m 2.0 ficytc_c + mql7_c --> 2.0 focytc_c + h_c + h_e + mqn7_c 566.1250665808258
FERCYT fe3_e + focytc_c <=> fe2_e + ficytc_c 1132.2501331616515
H2td h2_c <=> h2_e -591.6440432541838
HDH2 h2_c + 2.0 h_c + mqn7_c --> 2.0 h_e + mql7_c 556.4977639591415
EX_h_e h_e <=>  1118.7618026602795
EX_fe2_e fe2_e <=>  1132.2501331616515
EX_fe3_e  <=> fe3_e 1132.2501331616515
EX_h2_e h2_e <=>  -591.6440432541838


In [11]:
for name in new_names:
    print(f'"{name}"')

"Rhodoferax_mEmC_1_0.66.xml"
"Rhodoferax_mEmC_1_0.6.xml"
"Rhodoferax_mEmC_1_0.54.xml"
"Rhodoferax_mEmC_1_0.47.xml"
"Rhodoferax_mEmC_1_0.4.xml"
"Rhodoferax_mEmC_1_0.33.xml"
"Rhodoferax_mEmC_1_0.27.xml"
"Rhodoferax_mEmC_1_0.2.xml"
"Rhodoferax_mEmC_3_0.66.xml"
"Rhodoferax_mEmC_3_0.6.xml"
"Rhodoferax_mEmC_3_0.54.xml"
"Rhodoferax_mEmC_3_0.47.xml"
"Rhodoferax_mEmC_3_0.4.xml"
"Rhodoferax_mEmC_3_0.33.xml"
"Rhodoferax_mEmC_3_0.27.xml"
"Rhodoferax_mEmC_3_0.2.xml"
"Rhodoferax_mEmC_2_0.99.xml"
"Rhodoferax_mEmC_2_0.9.xml"
"Rhodoferax_mEmC_2_0.8.xml"
"Rhodoferax_mEmC_2_0.7.xml"
"Rhodoferax_mEmC_2_0.6.xml"
"Rhodoferax_mEmC_2_0.5.xml"
"Rhodoferax_mEmC_2_0.4.xml"
"Rhodoferax_mEmC_2_0.3.xml"
"Rhodoferax_mEmC_4_0.92.xml"
"Rhodoferax_mEmC_4_0.84.xml"
"Rhodoferax_mEmC_4_0.74.xml"
"Rhodoferax_mEmC_4_0.65.xml"
"Rhodoferax_mEmC_4_0.56.xml"
"Rhodoferax_mEmC_4_0.47.xml"
"Rhodoferax_mEmC_4_0.37.xml"
"Rhodoferax_mEmC_4_0.28.xml"
"Rhodoferax_mEmC_5_0.66.xml"
"Rhodoferax_mEmC_5_0.6.xml"
"Rhodoferax_mEmC_5_0.54.xml"

In [12]:
geo = cobra.io.read_sbml_model("models/Geobacter_mEmC_1_0.66.xml")
rho = cobra.io.read_sbml_model("models/Rhodoferax_mEmC_1_0.66.xml")

In [13]:
rho.summary()

Metabolite,Reaction,Flux,C-Number,C-Flux
fe2_e,EX_fe2_e,2.255E-05,0,0.00%
h_e,EX_h_e,50.02,0,0.00%
nh4_e,EX_nh4_e,7.236,0,0.00%
o2_e,EX_o2_e,0.3,0,0.00%
pi_e,EX_pi_e,0.6177,0,0.00%
so4_e,EX_so4_e,0.1658,0,0.00%
Metabolite,Reaction,Flux,C-Number,C-Flux
co2_e,EX_co2_e,-11.73,1,100.00%
h2o_e,EX_h2o_e,-9.542,0,0.00%


In [14]:
geo.summary()

Metabolite,Reaction,Flux,C-Number,C-Flux
ca2_e,EX_ca2_e,0.0006237,0,0.00%
fe3_e,EX_fe3_e,206.6,0,0.00%
h2_e,EX_h2_e,101.3,0,0.00%
k_e,EX_k_e,0.1472,0,0.00%
mg2_e,EX_mg2_e,0.02121,0,0.00%
nh4_e,EX_nh4_e,4.502,0,0.00%
pi_e,EX_pi_e,1.179,0,0.00%
so4_e,EX_so4_e,0.1108,0,0.00%
Metabolite,Reaction,Flux,C-Number,C-Flux
co2_e,EX_co2_e,-13.88,1,100.00%


In [15]:
geo.objective=geo.reactions.EX_fe3_e

In [16]:
geo.optimize('minimize')

,fluxes,reduced_costs
AACPAT,0.166625,0.00000
ACBIPGT,0.000000,0.00000
ACCOAC,2.827539,0.00000
ACGK,0.133681,0.00000
ACGS,0.000000,0.00000
...,...,...
EX_cit_e,0.000000,168.00000
EX_cd2_e,-0.000000,0.00000
EX_ca2_e,0.000624,0.00000
constraint_reaction,1.000000,413.25229
